In [ ]:
from scripts.sweep import run_task

server = "mine"

In [ ]:
# train fed diff os
from scripts.sweep import run_sweep

script = "train.os"
max_jobs = 1
grid = {
    "method": [f"client{i}_os" for i in range(5)]
}
run_sweep(script, server, max_jobs, grid)

In [ ]:
# syn ws
run_task("synthesis.ws", server, {"method": "ws"})

In [ ]:
# train cl diff by syn data
run_task("train.os", server, {"method": "cl_distill"})

In [ ]:
# train for model merging of ws
run_task("train.distill_noise", server)

In [ ]:
# syn for model merging of ws
run_task("synthesis.cl", server, {"method": "distill_direct_ddim_initclient_1_jumps_50_c5_a0.1_seed0_40"})

# Baselines

In [ ]:
# train cl
run_task("train.os", server, {"method": "cl"})

In [ ]:
# syn cl
run_task("synthesis.cl", server, {"method": "cl"})

In [ ]:
# train fed_diffuse
run_task("train.fed_diffuse", server)

In [ ]:
# syn fed_diffuse
run_task("synthesis.cl", server, {"method": "fed_diffuse"})

In [ ]:
# syn dfd
run_task("synthesis.dfd", server, {
    "num_samplings": 5000,
    "test_batchsize": 256
})

In [ ]:
# train server of pfdm
run_task("train.pfdm", server)

In [ ]:
# syn pfdm
run_task("synthesis.pfdm", server)

In [ ]:
# init codebooks for vq
run_task("train.vq", server, {"method": "codebook"})

In [ ]:
# train for vq
run_task("train.vq", server)

In [ ]:
# syn for vq
run_task("synthesis.vq", server)

# Test

In [ ]:
from synthesis.syn_utils import mv_checkpoints_to_model
mv_checkpoints_to_model(r"client\d+_os", 500)

In [ ]:
# Test
from config import args
from synthesis.syn_utils import compute_clean_fid
method = "fed_diffuse"
img_path = args.img_path / method
client_wise = False
if client_wise:
    for client in range(5):
        score = compute_clean_fid(img_path / str(client))
        print(f"Client {client} score: {score}")
else:
    score = compute_clean_fid(img_path)
    print(f"score: {score}")